# LC 104 — Maximum Depth of Binary Tree
**Day 36 | DFS on Binary Trees | Easy**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> The depth of any node is
<em>1 + the deeper of its two subtrees</em>. DFS naturally
computes this bottom-up — base case returns 0 at None,
each level adds 1.
</div>

## Official Problem Statement

Given the `root` of a binary tree, return *its maximum depth*.

A binary tree's **maximum depth** is the number of nodes along
the longest path from the root node down to the farthest leaf
node.

**Constraints:**
- The number of nodes in the tree is in the range `[0, 10^4]`.
- `-100 <= Node.val <= 100`

## What This Is Actually Asking

Count the longest root-to-leaf path measured in number of nodes.
An empty tree has depth 0; a single-node tree has depth 1.
We don't care about values — only structure matters here.
The answer is always at least 0 and at most the total number
of nodes (in a completely skewed tree).

## Walk Through an Example by Hand

Tree: `[3, 9, 20, None, None, 15, 7]`

```
    3
   / \
  9  20
     / \
    15   7
```

1. Call `maxDepth(3)` — has children, recurse both sides.
2. `maxDepth(9)` — both children None → return 1.
3. `maxDepth(20)` — recurse left and right.
   - `maxDepth(15)` → return 1
   - `maxDepth(7)`  → return 1
   - return 1 + max(1,1) = 2
4. Back at root: return 1 + max(1, 2) = **3**

## The Picture

DFS post-order traversal — answers bubble up from leaves:

```
        3          <- depth=3 (1 + max(1,2))
       / \
      9   20       <- 9:depth=1  20:depth=2
          / \
         15   7   <- both depth=1

DFS call stack (going down then back up):
  maxDepth(3)
    maxDepth(9)
      maxDepth(None) -> 0
      maxDepth(None) -> 0
    <- 1 + max(0,0) = 1
    maxDepth(20)
      maxDepth(15)
        maxDepth(None) -> 0
        maxDepth(None) -> 0
      <- 1
      maxDepth(7)
        maxDepth(None) -> 0
        maxDepth(None) -> 0
      <- 1
    <- 1 + max(1,1) = 2
  <- 1 + max(1,2) = 3  ✓
```

## When To Use This Pattern

- When you need a **scalar property** of a tree (height, size,
  sum), think recursive DFS returning a number.
- When the answer at a node **depends on both subtrees**,
  think post-order: compute children first, then combine.
- When the problem says "root to leaf", think DFS not BFS.
- When the tree can be empty, always handle `None` as base
  case returning a sensible identity value (0 for depth/sum).
- When asked for "maximum" or "minimum" across subtrees,
  think `1 + max(left, right)` pattern.

## The Approach

Use recursive DFS. If the node is None, the depth is 0 — that
is the base case. Otherwise recurse into both children and
take the max of their depths. Add 1 to account for the current
node. This visits every node exactly once giving O(n) time.

In [ ]:
from collections import deque
from typing import Optional


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def make_tree(vals):
    """BFS builder: construct TreeNode tree from level-order list."""
    if not vals:
        return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root

In [ ]:
def test_harness(func):
    """Run test cases for maxDepth."""
    cases = [
        # (tree_vals, expected, label)
        ([3, 9, 20, None, None, 15, 7], 3,
         "LC example 1"),
        ([1, None, 2],                  2,
         "LC example 2 — right-skewed"),
        ([],                            0,
         "Edge: empty tree"),
        ([1],                           1,
         "Edge: single node"),
        ([1, 2, 3, 4, None, None, None],4,
         "Left-heavy"),
    ]
    passed = 0
    for vals, expected, label in cases:
        root = make_tree(vals)
        result = func(root)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(f"{status} | {label} "
              f"| expected={expected} got={result}")
    print(f"\n{passed}/{len(cases)} tests passed.")


print("test_harness defined — run after implementing solution.")

In [ ]:
def maxDepth(root: Optional[TreeNode]) -> int:
    """
    Return the maximum depth of a binary tree.

    Strategy: Recursive DFS (post-order).
      - Base case: None  -> 0
      - Recurse into left and right subtrees.
      - Return 1 + max(left_depth, right_depth).

    Args:
        root: Root of the binary tree.

    Returns:
        Integer depth (0 for empty tree).

    Time:  O(n) — visits every node once.
    Space: O(h) — call stack; O(n) worst case skewed tree.
    """
    # Debug: show which node we are visiting
    print(f"  visiting node: {root.val if root else None}")

    # Base case
    # TODO: if not root ...

    # Recurse left subtree
    # TODO: left_depth = maxDepth(root.left)

    # Recurse right subtree
    # TODO: right_depth = maxDepth(root.right)

    # Debug: show computed depths before combining
    # print(f"  node={root.val} L={left_depth} R={right_depth}")

    # Combine and return
    # TODO: return 1 + max(left_depth, right_depth)

    pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(maxDepth)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute force (store all paths) | O(n) | O(n) | Unnecessary overhead |
| Recursive DFS (optimal) | O(n) | O(h) | h=height; O(log n) balanced |
| Iterative BFS | O(n) | O(w) | w=max width; often larger |

**n** = number of nodes, **h** = height of tree.

## Real World Connection

At **Citi**, a trade approval hierarchy is a tree — knowing the
maximum depth tells ops how many approval hops a complex
derivative requires, directly impacting SLA guarantees.
On **AWS Glue**, job dependency DAGs are trees; finding the
critical path length (analogous to max depth) determines the
minimum pipeline runtime.
As a **data engineer**, understanding tree depth is key when
reasoning about nested JSON schemas — deeply nested documents
blow up Spark's recursive schema inference, so you validate
depth before ingestion.
Monitoring tools also use tree depth to detect runaway
recursive query plans in databases.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra